### SQL Database

In [2]:
## Create sample like SQL Databse 

import sqlite3
import os 

os.makedirs("data/database", exist_ok=True)

In [3]:
conn = sqlite3.connect('data/database/sample.db')
cursor=conn.cursor()

In [4]:
cursor.execute('''CREATE TABLE IF NOT EXISTS employees (
                    id INTEGER PRIMARY KEY AUTOINCREMENT,
                    name TEXT NOT NULL,
                    age INTEGER NOT NULL,
                    department TEXT NOT NULL
                )''')

In [5]:
cursor.execute('''CREATE TABLE IF NOT EXISTS projects (
                    project_id INTEGER PRIMARY KEY AUTOINCREMENT,
                    project_name TEXT NOT NULL,
                    start_date TEXT NOT NULL,
                    end_date TEXT NOT NULL
                )''')

In [6]:
employees = [
    (1, 'Alice', 30, 'HR'),
    (2, 'Bob', 25, 'Engineering'),
    (3, 'Charlie', 35, 'Sales')
]

projects = [
    (1, 'Project A', '2023-01-01', '2023-06-30'),
    (2, 'Project B', '2023-02-15', '2023-07-15'),
    (3, 'Project C', '2023-03-10', '2023-08-10')
]

In [7]:
cursor.executemany('''INSERT INTO employees (id, name, age, department) VALUES (?, ?, ?, ?)''', employees)
cursor.executemany('''INSERT INTO projects (project_id, project_name, start_date, end_date) VALUES (?, ?, ?, ?)''', projects)


In [8]:
cursor.execute("SELECT * FROM employees")

In [10]:
conn.commit()
conn.close()

In [13]:
### Database Content Extraction 

from langchain_community.utilities import SQLDatabase
from langchain_community.document_loaders import SQLDatabaseLoader               

In [15]:
### Method1 SQLDATABASE UTILITY 

db = SQLDatabase.from_uri("sqlite:///data/database/sample.db")  

## get DATABASE INFO

print(f"Tables: {db.get_usable_table_names()}")
print(f"\nTable DDL:")
print(db.get_table_info())

Tables: ['employees', 'projects']

Table DDL:

CREATE TABLE employees (
	id INTEGER, 
	name TEXT NOT NULL, 
	age INTEGER NOT NULL, 
	department TEXT NOT NULL, 
	PRIMARY KEY (id)
)

/*
3 rows from employees table:
id	name	age	department
1	Alice	30	HR
2	Bob	25	Engineering
3	Charlie	35	Sales
*/


CREATE TABLE projects (
	project_id INTEGER, 
	project_name TEXT NOT NULL, 
	start_date TEXT NOT NULL, 
	end_date TEXT NOT NULL, 
	PRIMARY KEY (project_id)
)

/*
3 rows from projects table:
project_id	project_name	start_date	end_date
1	Project A	2023-01-01	2023-06-30
2	Project B	2023-02-15	2023-07-15
3	Project C	2023-03-10	2023-08-10
*/


In [29]:
## Method2 

from typing import List
from langchain_core.documents import Document 

# MMethod 2: Custom SQL to Document Conversation

print("\n\nMethod 2: Custom SQL to Document Conversation")

def sql_to_documents(db_path:str)->List[Document]:
    """Convert Sql Databse tp Documents with context"""
    conn=sqlite3.connect(db_path)
    cursor=conn.cursor()
    documents=[]

    # Stratergy 1: Create Documents for each table

    cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
    tables = cursor.fetchall()
    
    for table in tables:
        table_name=table[0]

        #Get Table SChemea
        cursor.execute(f"PRAGMA table_info({table_name});")
        columns = cursor.fetchall()
        columns_names = [col[1] for col in columns]

        #Get Table Content 

        cursor.execute(f"SELECT * FROM {table_name}")
        rows=cursor.fetchall()

        # Create table overview documents

        table_content = f"Table:{table_name}\n"
        table_content += f"columns:{','.join(columns_names)}\n"
        table_content += f"Total Records: {len(rows)}\n\n"

        # Add sample records

        table_content += "Sample Records:\n"
        for row in rows[:5]:
            record = dict(zip(columns_names,row))
            table_content += f"{record}\n"

        doc = Document(
            page_content=table_content,
            metadata={
                'source':db_path,
                'table_name':table_name,
                'num_records': len(rows),
                'data_type':'sql_table'
            }
        )
        documents.append(doc)
        
    return documents





Method 2: Custom SQL to Document Conversation


In [27]:
sql_to_documents("data/database/sample.db") 

[Document(metadata={'source': 'data/database/sample.db', 'table_name': 'employees', 'num_records': 3, 'data_type': 'sql_table'}, page_content="Table:employees\ncolumns:id,name,age,department\nTotal Records: 3\n\nSample Records:\n{'id': 1, 'name': 'Alice', 'age': 30, 'department': 'HR'}\n{'id': 2, 'name': 'Bob', 'age': 25, 'department': 'Engineering'}\n{'id': 3, 'name': 'Charlie', 'age': 35, 'department': 'Sales'}\n"),
 Document(metadata={'source': 'data/database/sample.db', 'table_name': 'sqlite_sequence', 'num_records': 2, 'data_type': 'sql_table'}, page_content="Table:sqlite_sequence\ncolumns:name,seq\nTotal Records: 2\n\nSample Records:\n{'name': 'employees', 'seq': 3}\n{'name': 'projects', 'seq': 3}\n"),
 Document(metadata={'source': 'data/database/sample.db', 'table_name': 'projects', 'num_records': 3, 'data_type': 'sql_table'}, page_content="Table:projects\ncolumns:project_id,project_name,start_date,end_date\nTotal Records: 3\n\nSample Records:\n{'project_id': 1, 'project_name': 